In [3]:
from heisenberg_hamiltonians import SpinSystem, HeisenbergJ1J2
from spin_lattices import KagomeLattice
import numpy as np
from typing import Callable
import numpy.typing as npt
import lattice_symmetries as ls

from scipy.sparse import csr_matrix, coo_matrix, diags
from scipy.sparse.csgraph import connected_components
from my_stopwatch import stopwatch
from scipy.special import logsumexp
from vmc_amplitude import compute_log_local_energies, true_relsigns, compute_local_energies_reference
from pathlib import Path
from slater_determinant import SlaterDeterminant, Initializer
import torch
import torch.nn as nn

In [ ]:
class AbsSlaterDet(nn.Module):
    def __init__(self, system: SpinSystem, initialization: str | Initializer = "orthogonal"):
        self.det = SlaterDeterminant(
            lattice=system.lattice,
            basis=system.canonical_basis,
            initialization=initialization,
            sign_cache_dir=Path("signs_cache"),
        )
        self.system = system

    def forward(self, states: torch.Tensor):
        indices = self.system.canonical_basis.index(states.detach().numpy().astype(np.uint64))
        return torch.abs(self.det(indices))
    